## Step 1: Environment Setup

In [ ]:
%%time
import os
import sys
import shutil
from pathlib import Path
import subprocess

# Configuration
REPO_INPUT = Path('/kaggle/input/gdsearch-repository')
WORKING_DIR = Path('/kaggle/working/GDSearch')
OUTPUT_DIR = Path('/kaggle/working/results')

print("="*80)
print("GDSearch Kaggle Environment Setup")
print("="*80)

# Check if repository exists
if not REPO_INPUT.exists():
    print("ERROR: Repository not found at /kaggle/input/gdsearch-repository")
    print("\nInstructions:")
    print("   1. Upload GDSearch repository as a Kaggle dataset")
    print("   2. Add dataset to this notebook")
    print("   3. Ensure it's mounted at /kaggle/input/gdsearch-repository")
    raise FileNotFoundError("GDSearch repository not found")

print(f"Repository found: {REPO_INPUT}")
print(f"Working directory: {WORKING_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

### Copy Repository to Working Directory

In [ ]:
%%time
print("Copying repository to working directory...")

# Remove existing working directory if present
if WORKING_DIR.exists():
    print(f"Removing existing {WORKING_DIR}")
    shutil.rmtree(WORKING_DIR)

# Copy repository
shutil.copytree(REPO_INPUT, WORKING_DIR, symlinks=False, ignore=None, dirs_exist_ok=True)
print(f"Repository copied to {WORKING_DIR}")

# Change to working directory
os.chdir(WORKING_DIR)
print(f"Current directory: {os.getcwd()}")

# Add to Python path - CRITICAL for src.* imports
if str(WORKING_DIR) not in sys.path:
    sys.path.insert(0, str(WORKING_DIR))
    print(f"Added {WORKING_DIR} to Python path")

# Verify Python path setup
print(f"\nPython path (first 3 entries):")
for i, p in enumerate(sys.path[:3], 1):
    print(f"  {i}. {p}")

# Verify key files
key_files = ['run_all_kaggle.py', 'requirements.txt', 'src/__init__.py']
for file in key_files:
    if (WORKING_DIR / file).exists():
        print(f"Found {file}")
    else:
        print(f"Missing {file}")

### Resume from Previous Results (Optional)

**If you have previous Kaggle run results:**

1. Upload previous `gdsearch_results_complete.zip` as a Kaggle dataset
2. Name it `result` and add it to this notebook
3. It should mount at `/kaggle/input/result/`
4. Set `RESUME_ENABLED = True` in the cell below
5. Run the cell to copy previous results
6. Experiments will automatically skip completed runs!

In [ ]:
%%time
# ============================================================================
# RESUME FROM PREVIOUS KAGGLE RUN (Optional)
# ============================================================================
# If you uploaded previous results as a Kaggle dataset at /kaggle/input/result,
# this cell will copy them to the working directory so experiments can resume

import shutil
from pathlib import Path

PREVIOUS_RESULTS = Path('/kaggle/input/result')
RESUME_ENABLED = False  # Set to True if you want to resume from previous results

if RESUME_ENABLED and PREVIOUS_RESULTS.exists():
    print("="*80)
    print("RESUMING FROM PREVIOUS RESULTS")
    print("="*80)
    print(f"Source: {PREVIOUS_RESULTS}")
    print(f"Destination: {OUTPUT_DIR}")
    
    # Copy all previous results to working directory
    # This includes experiments/, checkpoints/, visualizations/, etc.
    if (PREVIOUS_RESULTS / 'results_full').exists():
        print("\nCopying previous results...")
        shutil.copytree(PREVIOUS_RESULTS / 'results_full', OUTPUT_DIR / 'results_full', 
                       dirs_exist_ok=True)
        
        # Count copied files
        copied_files = sum(1 for _ in (OUTPUT_DIR / 'results_full').rglob('*') if _.is_file())
        print(f"✓ Copied {copied_files} files from previous run")
        
        # Show what's available to resume
        experiments_dir = OUTPUT_DIR / 'results_full' / 'experiments'
        if experiments_dir.exists():
            completed_exp = [d.name for d in experiments_dir.iterdir() if d.is_dir()]
            print(f"\nCompleted experiments found: {', '.join(completed_exp)}")
        
        checkpoints_dir = OUTPUT_DIR / 'results_full' / 'checkpoints'
        if checkpoints_dir.exists():
            checkpoint_count = len(list(checkpoints_dir.glob('*.pt')))
            print(f"Checkpoints found: {checkpoint_count} model files")
        
        print("\n" + "="*80)
        print("RESUME SETUP COMPLETE")
        print("="*80)
        print("Run experiments with --resume flag to skip completed experiments")
        print("="*80)
    else:
        print(f"\nWARNING: {PREVIOUS_RESULTS / 'results_full'} not found")
        print("Expected structure: /kaggle/input/result/results_full/")
        print("Please check your dataset upload structure")
        
elif RESUME_ENABLED:
    print("="*80)
    print("RESUME REQUESTED BUT NO PREVIOUS RESULTS FOUND")
    print("="*80)
    print(f"Looking for: {PREVIOUS_RESULTS}")
    print("\nTo use resume:")
    print("1. Upload previous results as a Kaggle dataset")
    print("2. Add it to this notebook")
    print("3. Ensure it's mounted at /kaggle/input/result")
    print("4. Set RESUME_ENABLED = True in this cell")
    print("="*80)
else:
    print("Resume disabled (RESUME_ENABLED = False)")
    print("Starting fresh experiments from scratch")
    print("To enable resume: Set RESUME_ENABLED = True and upload previous results")

### Install Dependencies

### NumPy/Pandas Compatibility Check (CRITICAL)

In [ ]:
%%time
import sys
import subprocess

print("Checking NumPy/Pandas compatibility...")
print("="*80)

def run_pip(args):
    """Run pip command and capture output"""
    cmd = [sys.executable, '-m', 'pip'] + args
    print('>', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

# Check current NumPy version
try:
    import numpy as np
    numpy_version = np.__version__
    numpy_major = int(numpy_version.split('.')[0])
    print(f"NumPy: {numpy_version} (will use this version)")
except Exception as e:
    print(f"NumPy import failed: {e}")
    raise RuntimeError("NumPy must be available")

# Check if Pandas can import successfully
pandas_import_error = None
try:
    import pandas as pd
    pandas_version = pd.__version__
    print(f"Pandas: {pandas_version} - imports successfully!")
    pandas_ok = True
except ValueError as e:
    if "numpy.dtype size changed" in str(e):
        print(f"Pandas import failed: Binary incompatibility with NumPy {numpy_version}")
        print(f"   Error: {e}")
        pandas_import_error = e
        pandas_ok = False
    else:
        raise
except Exception as e:
    print(f"Pandas import failed: {e}")
    pandas_import_error = e
    pandas_ok = False

# Decision: Fix Pandas to match NumPy (keep NumPy version as-is)
if not pandas_ok:
    print("\n" + "="*80)
    print("FIXING: Reinstalling Pandas to match NumPy {numpy_version}")
    print("="*80)
    print(f"Best Practice: We keep NumPy {numpy_version} (Kaggle's optimized version)")
    print(f"Solution: Reinstall Pandas with --no-cache-dir to rebuild against current NumPy")
    print("\nThis ensures binary compatibility without downgrading platform packages.")
    print("="*80)
    
    # Reinstall pandas (will rebuild/redownload wheel compatible with current numpy)
    rc = run_pip(['install', '--force-reinstall', '--no-cache-dir', '--no-deps', 'pandas'])
    
    # Reinstall pandas dependencies that may have been skipped
    if rc == 0:
        print("\nReinstalling Pandas dependencies...")
        run_pip(['install', 'pandas'])  # This installs missing deps without forcing reinstall
    
    if rc == 0:
        print("\n" + "="*80)
        print("PANDAS REINSTALLED SUCCESSFULLY")
        print("="*80)
        print("\nCRITICAL: You MUST restart the kernel now!")
        print("   1. Click 'Runtime' → 'Restart runtime' (or Kernel → Restart)")
        print("   2. Re-run ALL cells from the beginning")
        print("   3. Pandas C-extensions will load correctly after restart")
        print("\nDO NOT PROCEED without restarting!")
        print("="*80)
    else:
        print("\nPandas reinstall failed - check error output above")
        raise RuntimeError("Pandas compatibility fix failed")
else:
    print("\nNumPy and Pandas are compatible - no action needed!")
    print(f"   Using NumPy {numpy_version} and Pandas {pandas_version}")
    print("="*80)

In [ ]:
%%time
print("Installing dependencies...")
print("="*80)

# Check if requirements_kaggle.txt exists, otherwise use requirements.txt
if (WORKING_DIR / 'kaggle' / 'requirements_kaggle.txt').exists():
    requirements_file = WORKING_DIR / 'kaggle' / 'requirements_kaggle.txt'
    print(f"Using Kaggle-specific requirements: {requirements_file}")
elif (WORKING_DIR / 'requirements.txt').exists():
    requirements_file = WORKING_DIR / 'requirements.txt'
    print(f"Using standard requirements: {requirements_file}")
    print("   NOTE: This may overwrite NumPy/Pandas. Prefer kaggle/requirements_kaggle.txt")
else:
    raise FileNotFoundError("No requirements file found")

# Install dependencies (suppress most output, show only errors)
print("\nInstalling packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_file)],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Dependencies installed successfully")
    if result.stderr:
        # Show warnings but don't fail
        print("\nDependency warnings (usually safe to ignore):")
        # Filter out common non-critical warnings
        stderr_lines = result.stderr.split('\n')
        for line in stderr_lines[:20]:  # Show max 20 lines
            if line.strip() and 'incompatible' in line.lower():
                print(f"   {line}")
else:
    print("Dependency installation failed:")
    print(result.stderr)
    raise RuntimeError("Failed to install dependencies")

# Post-install verification
print("\nPost-install verification:")
critical_imports = [
    ('numpy', 'NumPy'),
    ('pandas', 'Pandas'),
    ('torch', 'PyTorch'),
    ('mlflow', 'MLflow'),
    ('optuna', 'Optuna'),
    ('transformers', 'Transformers'),
]

all_ok = True
for module_name, display_name in critical_imports:
    try:
        mod = __import__(module_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"   {display_name}: {version}")
    except Exception as e:
        print(f"   {display_name}: {e}")
        all_ok = False

if not all_ok:
    print("\nSome critical packages failed to import!")
    print("   Try manually installing the missing package(s) and re-running this cell.")
    raise RuntimeError("Critical import failures detected")

print("\n" + "="*80)
print("All critical dependencies verified!")
print("="*80)

# CRITICAL: Verify src.* imports work (prevents csv_utils import errors)
print("\nVerifying GDSearch module imports...")
try:
    from src.utils.csv_utils import safe_read_csv
    print("   ✅ src.utils.csv_utils - OK")
except ImportError as e:
    print(f"   ❌ src.utils.csv_utils - FAILED: {e}")
    print(f"\n   Current directory: {os.getcwd()}")
    print(f"   Python path: {sys.path[:3]}")
    print(f"   WORKING_DIR in path: {str(WORKING_DIR) in sys.path}")
    raise RuntimeError("GDSearch module imports failed - check Python path setup")

try:
    from src.core.experiment_tracker import ExperimentTracker
    print("   ✅ src.core.experiment_tracker - OK")
except ImportError as e:
    print(f"   ❌ src.core.experiment_tracker - FAILED: {e}")
    raise RuntimeError("GDSearch module imports failed")

print("   All GDSearch modules verified!")
print("="*80)

In [ ]:
%%time
print("PRE-DOWNLOADING ALL DATASETS (CRITICAL for Kaggle time savings!)")
print("="*80)
print("This step downloads all datasets ONCE instead of repeatedly during experiments.")
print("Saves 30-60 minutes of Kaggle runtime!\n")

try:
    # Run the dataset download script
    result = subprocess.run(
        [sys.executable, 'download_datasets_kaggle.py'],
        capture_output=True,
        text=True,
        cwd=WORKING_DIR
    )
    
    # Show output
    print(result.stdout)
    if result.returncode != 0:
        print("\nDataset download warnings (non-critical):")
        print(result.stderr)
    
    print("\n" + "="*80)
    print("DATASET PRE-DOWNLOAD COMPLETE")
    print("="*80)
    print("All datasets cached and ready for experiments!")
    print("Experiments will run MUCH faster now.")
    print("="*80)
    
except Exception as e:
    print(f"\nDataset download failed: {e}")
    print("Experiments will download datasets on-demand (slower but still works)")
    print("="*80)

### Download Datasets

**Important:** Datasets will be downloaded automatically when experiments run, but you can pre-download them here to verify connectivity.

### Verify Environment

In [ ]:
print("Verifying environment...")
print("="*80)

# Check Python version
print(f"Python: {sys.version.split()[0]}")

# Check PyTorch and CUDA
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - CUDA version: {torch.version.cuda}")
    print(f"   - GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"     Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")

# Check key dependencies
try:
    import numpy as np
    print(f"NumPy: {np.__version__}")
except ImportError as e:
    print(f"NumPy: {e}")

try:
    import pandas as pd
    print(f"Pandas: {pd.__version__}")
except ImportError as e:
    print(f"Pandas: {e}")

try:
    import matplotlib
    print(f"Matplotlib: {matplotlib.__version__}")
except ImportError as e:
    print(f"Matplotlib: {e}")

try:
    import tqdm
    print(f"tqdm: {tqdm.__version__}")
except ImportError as e:
    print(f"tqdm: {e}")

try:
    import mlflow
    print(f"MLflow: {mlflow.__version__}")
except ImportError as e:
    print(f"MLflow: {e}")

# Verify GDSearch modules (ONLY after Python path is set up in Cell 4)
try:
    from src.core import optimizers
    from src.utils.csv_utils import safe_read_csv
    print(f"✓ GDSearch core modules imported successfully")
    print(f"✓ CSV utilities available")
except ImportError as e:
    print(f"❌ GDSearch import error: {e}")
    print(f"   Make sure Cell 4 (Python path setup) was executed first!")

print("="*80)
print("Environment setup complete!")

In [ ]:
# =============================================================================
# GPU DETECTION AND PARALLEL MODE CONFIGURATION
# =============================================================================
print("Detecting GPU Configuration...")
print("="*80)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Available GPUs: {gpu_count}")

if gpu_count >= 2:
    print("\n✅ MULTI-GPU DETECTED - Parallel Experiments Enabled!")
    print("\nGPU Details:")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"      Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"      Compute Capability: {props.major}.{props.minor}")
    
    PARALLEL_EXPERIMENTS = True
    print(f"\n🚀 Parallel Mode: ENABLED")
    print(f"   - Will run 2 experiments simultaneously (one per GPU)")
    print(f"   - Expected speedup: ~2x faster than sequential mode")
    print(f"   - GPU utilization: ~100% (both GPUs active)")
    
elif gpu_count == 1:
    print(f"\nSingle GPU mode: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"   Memory: {props.total_memory / 1024**3:.2f} GB")
    print(f"   Compute Capability: {props.major}.{props.minor}")
    
    PARALLEL_EXPERIMENTS = False
    print(f"\nℹ️  Sequential mode (1 GPU)")
    print(f"   - Experiments run one at a time")
    print(f"   - To enable parallel mode: Use Kaggle T4x2 or P100x2")
else:
    print("\n⚠️  No GPU detected - CPU mode")
    PARALLEL_EXPERIMENTS = False

print("="*80)

### Verify Audit Fixes (Label Smoothing, AMP, EMA)

**NEW - January 2026:** Verify that the audit fixes are integrated and available.

## ⚠️ Parallel Execution: Infrastructure Exists, Integration Pending

**STATUS UPDATE - February 2026:** Parallel execution infrastructure is built but NOT yet integrated in main execution.

### What's Available Now
- ✅ `ParallelExperimentRunner` module exists and is tested
- ✅ `--parallel` and `--num-gpus` CLI flags are defined
- ✅ GPU detection and worker pool logic implemented
- ❌ **NOT integrated in run_all_kaggle.py main() function**

### Current Behavior (All Instances)

| Instance Type | GPUs | Actual Mode | Time for 'all' | GPU Utilization |
|---------------|------|-------------|----------------|-----------------|
| T4 (standard) | 1 | Sequential | ~12 hours | Uses GPU 0 only |
| T4x2 | 2 | **Sequential** | **~12 hours** | **Uses GPU 0 only (GPU 1 idle)** |
| P100x2 | 2 | **Sequential** | **~12 hours** | **Uses GPU 0 only (GPU 1 idle)** |

### Why Sequential Mode Only?
The parallel infrastructure exists but is not connected:
- CLI flags are defined but never checked in main()
- `ParallelExperimentRunner` is never instantiated
- All experiment loops use sequential execution
- **This is safe and stable** - sequential mode is fully tested

### Future Integration (Planned)
To enable parallel execution, the following integration is needed:
```python
# In run_all_kaggle.py main() after argparse:
if args.parallel:
    runner = ParallelExperimentRunner(num_gpus=args.num_gpus)
    results = runner.run_experiments(experiment_configs)
```

### Recommendation
- Use **any GPU instance** - all will run sequentially
- T4x2 won't provide speedup until parallel mode is integrated
- Sequential mode is stable and tested - no functionality is lost

In [ ]:
print("Verifying Audit Fixes Integration...")
print("="*80)

# Check that audit fix files exist
audit_fix_files = [
    ('configs/label_smoothing_ablation.json', 'Label smoothing ablation config'),
    ('tests/test_integration_label_smoothing.py', 'Integration tests'),
    ('scripts/validate_audit_fixes.py', 'Validation script'),
    ('docs/LABEL_SMOOTHING_IMPLEMENTATION.md', 'Documentation'),
    ('docs/AUDIT_FIX_REPORT.md', 'Audit report')
]

all_present = True
for file_path, description in audit_fix_files:
    full_path = WORKING_DIR / file_path
    if full_path.exists():
        print(f"✓ {description}: Found")
    else:
        print(f"✗ {description}: MISSING at {file_path}")
        all_present = False

# Check that core modules have the audit fix functions
print("\nVerifying core module functions...")
try:
    from src.core.training_utils import (
        LabelSmoothingCrossEntropy,
        AMPWrapper,
        ModelEMA,
        get_loss_function,
        create_amp_wrapper,
        create_model_ema
    )
    print("✓ Label smoothing: LabelSmoothingCrossEntropy available")
    print("✓ AMP: AMPWrapper and create_amp_wrapper available")
    print("✓ EMA: ModelEMA and create_model_ema available")
    print("✓ Loss factory: get_loss_function available")
    
    # Test that loss function works with label smoothing
    loss_fn = get_loss_function('cross_entropy', label_smoothing=0.1)
    entropy_floor = loss_fn.get_entropy_floor(10)
    print(f"✓ Label smoothing entropy floor calculation works: {entropy_floor:.4f}")
    
except ImportError as e:
    print(f"✗ Import error: {e}")
    all_present = False

# Check run_nn_experiment.py has audit fix integration
print("\nVerifying training pipeline integration...")
run_nn_file = WORKING_DIR / 'src' / 'experiments' / 'run_nn_experiment.py'
if run_nn_file.exists():
    content = run_nn_file.read_text(encoding='utf-8')
    
    checks = [
        ('get_loss_function', 'Loss function factory import'),
        ('AMPWrapper', 'AMP wrapper import'),
        ('ModelEMA', 'EMA model import'),
        ('create_amp_wrapper', 'AMP factory import'),
        ('create_model_ema', 'EMA factory import'),
        ('label_smoothing', 'Label smoothing config usage'),
        ('use_amp', 'AMP config usage'),
        ('use_ema', 'EMA config usage'),
        ('ema.shadow', 'EMA shadow model evaluation')
    ]
    
    for check_str, description in checks:
        if check_str in content:
            print(f"✓ {description}: Integrated")
        else:
            print(f"✗ {description}: NOT FOUND")
            all_present = False
else:
    print(f"✗ Training pipeline not found: {run_nn_file}")
    all_present = False

print("\n" + "="*80)
if all_present:
    print("✅ ALL AUDIT FIXES VERIFIED AND READY!")
    print("\nThe following features are now available in experiments:")
    print("  - Label smoothing (configurable, default=0.0)")
    print("  - AMP (automatic mixed precision, enabled by --kaggle-t4)")
    print("  - EMA (exponential moving average, configurable)")
    print("\nRun experiments with 'all' to include missing_ablations experiment")
    print("which contains the label_smoothing_ablation analysis.")
else:
    print("⚠️  SOME AUDIT FIXES MISSING!")
    print("\nThis may happen if:")
    print("  - Repository was uploaded without recent changes")
    print("  - Files were not committed to Git")
    print("\nExperiments will still run but may not include audit fix features.")
print("="*80)

In [ ]:
print("Verifying Parallel Execution Support...")
print("="*80)

# IMPORTANT: Parallel execution infrastructure exists but is NOT YET INTEGRATED
# The --parallel and --num-gpus flags are defined in run_all_kaggle.py argparse
# but the main() function does not yet implement parallel execution logic.
# 
# This is a FUTURE FEATURE - infrastructure exists but integration is incomplete.

# Check if parallel runner module exists
parallel_runner_file = WORKING_DIR / 'src' / 'utils' / 'parallel_experiment_runner.py'
if parallel_runner_file.exists():
    print("✓ Parallel runner module: Found (infrastructure exists)")
    print("⚠️  NOTE: Parallel execution is NOT YET INTEGRATED in run_all_kaggle.py")
    print("          The module exists but main() does not invoke it.")
    print("          Future feature - will be integrated in upcoming releases.")
    
    # For now, always use sequential mode
    PARALLEL_EXPERIMENTS = False
    
    print("\nℹ️  Using Sequential Execution")
    print(f"   - Single GPU mode: Experiments run one at a time")
    print(f"   - Parallel mode coming soon (infrastructure ready)")
else:
    print(f"✗ Parallel runner not found: {parallel_runner_file}")
    PARALLEL_EXPERIMENTS = False
    print("   Will use sequential execution")

print("="*80)

## Step 3: Run Experiments

## ⚠️ Parallel Execution Status: PLANNED BUT NOT YET IMPLEMENTED

**CURRENT STATUS: Infrastructure exists but NOT integrated in main execution**

### What Exists
- ✅ `src/utils/parallel_experiment_runner.py` - Parallel runner module (323 lines)
- ✅ `--parallel` and `--num-gpus` CLI flags defined in argparse
- ✅ GPU detection and allocation logic
- ✅ Queue-based worker pool implementation

### What's Missing
- ❌ **Integration in run_all_kaggle.py main()** - The critical missing piece
- ❌ Main execution loop does NOT check `args.parallel` flag
- ❌ ParallelExperimentRunner is never instantiated or invoked
- ❌ All experiments run sequentially regardless of flag

### Current Behavior
| Instance Type | GPUs | Mode | Actual Behavior |
|---------------|------|------|-----------------|
| T4 (standard) | 1 | Sequential | ✅ Uses GPU 0, works correctly |
| T4x2 | 2 | **Sequential** | ⚠️ Only uses GPU 0 (GPU 1 idle) |
| P100x2 | 2 | **Sequential** | ⚠️ Only uses GPU 0 (GPU 1 idle) |

### Why This Happened
This is a case of **documentation ahead of implementation**:
- Infrastructure was built and tested in isolation
- CLI flags were added in preparation
- Documentation was written assuming integration
- **Integration step was not completed**

### Workaround (Current)
- Use sequential execution (default behavior)
- All experiments run on single GPU
- No performance penalty vs. intended design (same result, just slower)

### Roadmap (Future)
To enable parallel execution, need to:
1. Add `if args.parallel:` check in main() after argparse
2. Instantiate `ParallelExperimentRunner(num_gpus=args.num_gpus)`  
3. Replace sequential experiment loop with parallel runner
4. Handle result collection from parallel workers

**For now: All experiments run sequentially, which is stable and tested.**

In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION - FULL MODE FORCED
# =============================================================================

# ===== Option 1: Quick Test (5 minutes) =====
# Fast smoke test - 2 epochs, 3 seeds, MNIST only
# EXPERIMENT_MODE = 'quick'
# EXPERIMENTS = 'mnist'
# SEEDS = '42,123,456'
# EXTRA_ARGS = ['--ultra-quick', '--robust-gradients', '--grad-noise-every', '0']

# ===== Option 2: Medium Test (30 minutes) =====
# Comprehensive test - 10 epochs, 3 seeds, MNIST + CIFAR-10
# EXPERIMENT_MODE = 'medium'
# EXPERIMENTS = 'mnist,cifar10'
# SEEDS = '42,123,456'
# EXTRA_ARGS = ['--quick', '--robust-gradients']

# ===== Option 3: Full Production Run (12 hours on T4) =====
# Complete benchmark - all experiments, 10 seeds
EXPERIMENT_MODE = 'full'
EXPERIMENTS = 'all'
SEEDS = '42,123,456,789,1011,1213,1415,1617,1819,2021'
EXTRA_ARGS = ['--robust-gradients', '--kaggle-t4']

# =============================================================================
# GPU & PARALLEL CONFIGURATION
# =============================================================================

# Detect available GPUs
import torch
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0

print(f"GPU Configuration: {gpu_count} GPU(s) available")
if gpu_count > 0:
    for i in range(gpu_count):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

# IMPORTANT: Parallel execution is NOT YET IMPLEMENTED in run_all_kaggle.py
# Even though infrastructure exists, the main() function does not use it.
# Always use sequential mode for now.
PARALLEL_EXPERIMENTS = False

if gpu_count >= 2:
    print("\n⚠️  NOTE: Multi-GPU detected but parallel mode NOT YET AVAILABLE")
    print("   Reason: Parallel infrastructure exists but not integrated in main()")
    print("   Status: Future feature - currently experiments run sequentially")
    print("   Impact: All experiments will use GPU 0 only")
elif gpu_count == 1:
    print("\nℹ️  Single GPU detected - Sequential execution")
    print("   All experiments will run one at a time on GPU 0")
else:
    print("\n⚠️  No GPU detected - Using CPU (will be very slow)")

# Resume configuration
RESUME_ENABLED = False  # Set to True to resume from partial results

print("\n" + "="*80)
print(f"EXPERIMENT CONFIGURATION SUMMARY")
print("="*80)
print(f"Mode: {EXPERIMENT_MODE.upper()}")
print(f"Experiments: {EXPERIMENTS}")
print(f"Seeds: {SEEDS}")
print(f"Extra Args: {' '.join(EXTRA_ARGS)}")
print(f"Parallel Mode: {'DISABLED (sequential)' if not PARALLEL_EXPERIMENTS else 'ENABLED'}")
print(f"Resume: {'Enabled' if RESUME_ENABLED else 'Disabled'}")
print("="*80)

### Verify Bug Fixes (February 2026)

In [ ]:
print("Verifying Bug Fixes (February 2026)...")
print("="*80)

# Check critical bug fixes
bug_fixes_verified = []

# 1. Check queue import in parallel runner (infrastructure exists)
try:
    from src.utils.parallel_experiment_runner import ParallelExperimentRunner
    import inspect
    source = inspect.getsource(ParallelExperimentRunner._worker)
    
    if 'import queue' in source or 'queue.Empty' in source:
        bug_fixes_verified.append("✅ Bug #1: queue.Empty exception handling fixed (infrastructure)")
    else:
        bug_fixes_verified.append("⚠️  Bug #1: queue import not found (check implementation)")
except Exception as e:
    bug_fixes_verified.append(f"❌ Bug #1 check failed: {e}")

# 2. Check CUDA_VISIBLE_DEVICES order (infrastructure exists)
try:
    from src.utils.parallel_experiment_runner import run_experiment_on_gpu
    import inspect
    source = inspect.getsource(run_experiment_on_gpu)
    
    # Check if env set before cuda ops
    env_idx = source.find("os.environ")
    cuda_idx = source.find("torch.cuda")
    
    if env_idx > 0 and cuda_idx > 0 and env_idx < cuda_idx:
        bug_fixes_verified.append("✅ Bug #2: CUDA device isolation fixed (infrastructure)")
    else:
        bug_fixes_verified.append("⚠️  Bug #2: Manual verification needed")
except Exception as e:
    bug_fixes_verified.append(f"❌ Bug #2 check failed: {e}")

# 3. Check run_experiment function exists
try:
    from src.experiments.run_nn_experiment import run_experiment
    bug_fixes_verified.append("✅ Bug #3: run_experiment() wrapper exists")
except ImportError:
    bug_fixes_verified.append("❌ Bug #3: run_experiment() missing")

# 4. Check Windows atomic rename
try:
    from src.utils.checkpoint_utils import save_checkpoint_atomic
    import inspect
    source = inspect.getsource(save_checkpoint_atomic)
    
    if 'os.name' in source and ('MoveFileExW' in source or 'windll' in source):
        bug_fixes_verified.append("✅ Bug #4: Windows atomic rename implemented")
    else:
        bug_fixes_verified.append("⚠️  Bug #4: Windows atomic rename not found (check implementation)")
except Exception as e:
    bug_fixes_verified.append(f"❌ Bug #4 check failed: {e}")

# 5. Check bias correction fix (optional - optimizer implementation detail)
try:
    from src.core.optimizers import AdamW
    import inspect
    source = inspect.getsource(AdamW._step_array)
    
    if 'max_safe_t' in source or 'adaptive' in source.lower():
        bug_fixes_verified.append("✅ Bug #5: AdamW bias correction underflow fixed")
    else:
        bug_fixes_verified.append("⚠️  Bug #5: Bias correction fix not found (non-critical)")
except Exception as e:
    bug_fixes_verified.append(f"⚠️  Bug #5 check skipped: {e}")

# 6. Check --parallel flag exists (BUT NOT YET INTEGRATED)
try:
    run_all_file = WORKING_DIR / 'run_all_kaggle.py'
    if run_all_file.exists():
        content = run_all_file.read_text()
        if '--parallel' in content and 'action=' in content:
            bug_fixes_verified.append("✅ CLI: --parallel flag defined (NOT yet integrated in main())")
        else:
            bug_fixes_verified.append("❌ CLI: --parallel flag not found")
except Exception as e:
    bug_fixes_verified.append(f"❌ CLI check failed: {e}")

print("\nInfrastructure Verification Results:")
print("="*80)
for result in bug_fixes_verified:
    print(result)

print("\n" + "="*80)
print("⚠️  IMPORTANT: Parallel execution infrastructure EXISTS but NOT INTEGRATED")
print("   - All required modules are present and tested")
print("   - CLI flags are defined in argparse")
print("   - BUT: main() does not invoke parallel runner")
print("   - Result: All experiments run sequentially (stable, tested)")
print("\n   This notebook will run successfully in SEQUENTIAL mode.")
print("="*80)

### Execute Experiments

In [ ]:
%%time
print("="*80)
print(f"Starting {EXPERIMENT_MODE.upper()} mode experiments")
print("="*80)

# Build command WITHOUT parallel flags (not yet implemented)
cmd = [
    sys.executable,
    'run_all_kaggle.py',
    '--experiments', EXPERIMENTS,
    '--seeds', SEEDS,
    '--results-dir', str(RESULTS_DIR),
    '--no-mlflow'  # CRITICAL: Disable MLflow in Kaggle (DB schema issues + read-only filesystem)
] + EXTRA_ARGS

# NOTE: --parallel and --num-gpus flags are NOT added because parallel execution
# is not yet integrated in run_all_kaggle.py main() function.
# Even though the flags are defined, main() does not check or use them.

print("\nℹ️  Sequential mode: Experiments run one at a time")
print("   - All experiments will use GPU 0")
if gpu_count >= 2:
    print(f"   - NOTE: {gpu_count} GPUs detected but only GPU 0 will be used")
    print("   - Parallel mode is planned but not yet integrated")

# Add --resume flag if previous results were loaded
if RESUME_ENABLED and (OUTPUT_DIR / 'results_full').exists():
    cmd.append('--resume')
    print("\n🔄 RESUME MODE ENABLED - Will skip completed experiments")
    print("="*80)

# Show what command will actually be executed
print("\n" + "="*80)
print("COMMAND TO BE EXECUTED:")
print("="*80)
print(" ".join(cmd))
print("="*80)

# Show experiment scope
exp_list = EXPERIMENTS.split(',')
seed_list = SEEDS.split(',')
print(f"\nExperiment Scope:")
print(f"   Experiments: {len(exp_list)} types")
print(f"   Seeds: {len(seed_list)} seeds per experiment")
print(f"   Total runs: ~{len(exp_list) * len(seed_list)} individual runs")
print(f"   Mode: Sequential (one GPU, one at a time)")

# Estimate time
if '--ultra-quick' in EXTRA_ARGS:
    est_time = "5-10 minutes"
elif '--quick' in EXTRA_ARGS:
    est_time = "30-60 minutes"
else:
    est_time = "8-12 hours"
print(f"   Estimated time: {est_time}")

# Show flags explanation
print("\n" + "="*80)
print("FLAG EXPLANATIONS:")
print("="*80)
for arg in cmd[3:]:  # Skip python, script, and base args
    if arg.startswith('--'):
        if arg == '--no-mlflow':
            print(f"  {arg}: Disable MLflow (Kaggle compatibility)")
        elif arg == '--ultra-quick':
            print(f"  {arg}: 2 epochs only (fast smoke test)")
        elif arg == '--quick':
            print(f"  {arg}: Reduced epochs (fast validation)")
        elif arg == '--kaggle-t4':
            print(f"  {arg}: T4 GPU optimizations (larger batch, mixed precision)")
        elif arg == '--robust-gradients':
            print(f"  {arg}: Enable AGC + gradient monitoring")
        elif arg == '--resume':
            print(f"  {arg}: Skip already completed experiments")
        elif arg == '--results-dir':
            print(f"  {arg}: Output directory for results")
        elif arg == '--experiments':
            print(f"  {arg}: Which experiment types to run")
        elif arg == '--seeds':
            print(f"  {arg}: Random seeds for reproducibility")

print("="*80)
print("\n⏳ Starting experiments... (this will take a while)")
print("="*80 + "\n")

# Execute the command
import subprocess
import sys

try:
    result = subprocess.run(cmd, check=True, capture_output=False, text=True)
    print("\n" + "="*80)
    print("✅ EXPERIMENTS COMPLETED SUCCESSFULLY")
    print("="*80)
except subprocess.CalledProcessError as e:
    print("\n" + "="*80)
    print(f"❌ EXPERIMENTS FAILED (exit code: {e.returncode})")
    print("="*80)
    print("\nCheck error messages above for details.")
    raise
except KeyboardInterrupt:
    print("\n" + "="*80)
    print("⚠️  EXPERIMENTS INTERRUPTED BY USER")
    print("="*80)
    print("\nPartial results may be available in results directory.")
    raise

## Step 4: Results Analysis

### List Generated Results

In [ ]:
import os
from pathlib import Path

print("Generated Results:")
print("="*80)

# List all result directories
result_dirs = [
    'experiments',
    '2d_optimization',
    'beta_sensitivity',
    'hyperparameter_sensitivity',
    'theory_practice',
    'visualizations',
    'analysis',
    'reports'
]

for dir_name in result_dirs:
    dir_path = RESULTS_DIR / dir_name
    if dir_path.exists():
        file_count = sum(1 for _ in dir_path.rglob('*') if _.is_file())
        print(f"{dir_name}: {file_count} files")
        
        # Show first few files
        files = sorted(dir_path.rglob('*.csv'))[:5]
        if files:
            for f in files:
                rel_path = f.relative_to(RESULTS_DIR)
                print(f"   - {rel_path}")
            if len(list(dir_path.rglob('*.csv'))) > 5:
                print(f"   ... and {len(list(dir_path.rglob('*.csv'))) - 5} more CSV files")
    else:
        print(f"{dir_name}: not found")

print("="*80)

### Quick Results Preview

In [ ]:
import pandas as pd
import glob

print("Quick Results Preview:")
print("="*80)

# Find MNIST results
# Note: safe_read_csv already imported in verification cell
mnist_csvs = list((RESULTS_DIR / 'experiments' / 'mnist').glob('*.csv'))

if mnist_csvs:
    print(f"\nFound {len(mnist_csvs)} MNIST result files\n")
    
    # Load and display summary
    results = []
    for csv in mnist_csvs[:10]:  # Show first 10
        df = safe_read_csv(csv)
        if df is None or len(df) == 0:
            print(f"Skipping empty or unreadable file {csv}")
            continue
        if len(df) > 0:
            final_row = df.iloc[-1]
            results.append({
                'file': csv.name,
                'epochs': len(df),
                'final_train_loss': final_row.get('train_loss', 'N/A'),
                'final_test_acc': final_row.get('test_acc', 'N/A'),
                'final_grad_norm': final_row.get('grad_norm', 'N/A')
            })
    
    if results:
        summary_df = pd.DataFrame(results)
        print(summary_df.to_string(index=False))
        
        # Check for grad_norm column
        if 'final_grad_norm' in summary_df.columns:
            has_grad_norm = summary_df['final_grad_norm'] != 'N/A'
            if has_grad_norm.all():
                print("\nAll results include gradient norm tracking!")
            else:
                print("\nSome results missing gradient norm")
else:
    print("No MNIST results found")

print("\n" + "="*80)

### Display Visualizations

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
import glob

print("Visualizations:")
print("="*80)

# Find visualization PNGs
viz_dir = RESULTS_DIR / 'visualizations' / 'static'
if viz_dir.exists():
    png_files = sorted(viz_dir.rglob('*.png'))[:5]  # Show first 5
    
    if png_files:
        for png in png_files:
            print(f"\n{png.name}")
            try:
                display(Image(filename=str(png), width=800))
            except Exception as e:
                print(f"   Could not display: {e}")
    else:
        print("No visualization PNGs found")
else:
    print("Visualization directory not found")

print("\n" + "="*80)

## Step 5: Save Results for Download

In [ ]:
%%time
print("Preparing results for download...")
print("="*80)

# Create archive
import shutil
archive_name = f'gdsearch_results_{EXPERIMENT_MODE}'
archive_path = OUTPUT_DIR / archive_name

print(f"Creating archive: {archive_name}.zip")
shutil.make_archive(str(archive_path), 'zip', RESULTS_DIR)

# Get archive size
archive_file = f"{archive_path}.zip"
size_mb = os.path.getsize(archive_file) / (1024 * 1024)
print(f"Archive created: {size_mb:.2f} MB")

# Summary
print("\n" + "="*80)
print("Results saved to:")
print(f"   - Directory: {RESULTS_DIR}")
print(f"   - Archive: {archive_file}")
print("\nTo download:")
print("   1. Check the 'Output' tab in Kaggle")
print(f"   2. Download {archive_name}.zip")
print("   3. Extract and analyze locally")
print("="*80)

## Step 6: Experiment Summary Report

In [ ]:
# Check for auto-generated summary report
summary_report = RESULTS_DIR / 'reports' / '00_EXPERIMENT_SUMMARY.md'

print("Experiment Summary:")
print("="*80)

if summary_report.exists():
    with open(summary_report, 'r') as f:
        print(f.read())
else:
    print("Summary report not found")
    print("\nManual Summary:")
    print(f"- Mode: {EXPERIMENT_MODE}")
    print(f"- Experiments: {EXPERIMENTS}")
    print(f"- Seeds: {SEEDS}")
    print(f"- Results directory: {RESULTS_DIR}")

print("\n" + "="*80)

---

## Completion Checklist

After running this notebook, verify:

- [ ] Environment setup completed without errors
- [ ] Quick validation test passed
- [ ] Experiments ran successfully
- [ ] Results generated in expected directories
- [ ] CSV files contain required columns (grad_norm, test_acc, etc.)
- [ ] Visualizations generated (if applicable)
- [ ] Results archive created for download

---

## Troubleshooting

### Common Issues:

**1. Repository not found**
```python
# Check dataset mounting:
!ls /kaggle/input/
```

**2. Out of Memory (OOM)**
```python
# Reduce batch size or use ultra-quick mode
EXTRA_ARGS = ['--ultra-quick', '--batch-size', '32']
```

**3. Time limit exceeded**
```python
# Reduce time budget or number of seeds
SEEDS = '42,123,456'  # Use fewer seeds
EXTRA_ARGS = ['--time-budget', '3.0']  # Lower budget
```

**4. Missing dependencies**
```python
# Manually install missing package
!pip install <package-name>
```

---

## Additional Resources

- **Documentation:** See `README.md` in repository
- **Proposal Compliance:** See `docs/PROPOSAL_COMPLIANCE_CHECKLIST.md`
- **Configuration Schema:** See `configs/config_schema.json`

---

**Generated by GDSearch Kaggle Runner**  
*Last Updated: December 24, 2025*

## Quick Download Results (Add this link)

In [ ]:
# ============================================================================
# DOWNLOAD ALL RESULTS (Everything including checkpoints)
# ============================================================================

from IPython.display import FileLink
import shutil
import os

print("Creating downloadable archive of ALL RESULTS...")
print("="*80)

# Archive the entire results directory (including checkpoints)
archive_path = '/kaggle/working/gdsearch_results_complete'
shutil.make_archive(archive_path, 'zip', RESULTS_DIR)

# Get archive size
archive_size_mb = os.path.getsize(f'{archive_path}.zip') / (1024**2)

# Count files
file_count = sum(1 for _ in RESULTS_DIR.rglob('*') if _.is_file())

print("\n" + "="*80)
print("✅ COMPLETE RESULTS ARCHIVE CREATED")
print("="*80)
print(f"📦 Archive: {archive_path}.zip")
print(f"📊 Size: {archive_size_mb:.2f} MB")
print(f"📁 Files: {file_count}")
print(f"📂 Includes: experiments, checkpoints, visualizations, reports, analysis")
print(f"\n📥 Download from Kaggle Output tab or click link below:")
print("="*80)

# Display download link
FileLink(f'{archive_path}.zip')

## Quick Download Results

In [ ]:
# QUICK DOWNLOAD: Click the folder icon and download results manually
# OR use this command to create a downloadable archive:

#from IPython.display import FileLink
#import shutil

#print("Creating downloadable results archive...")
#archive_path = '/kaggle/working/results_download'
#shutil.make_archive(archive_path, 'zip', RESULTS_DIR)

#print(f"\nResults archived: {archive_path}.zip")
#print(f"Size: {os.path.getsize(f'{archive_path}.zip') / (1024**2):.2f} MB")
#print("\n📥 Download from Output tab or use link below:")

# Create download link
#FileLink(f'{archive_path}.zip')